# N2/N6: chạy RTG (bản B, C, D) cho CUB, Stanford Cars, FGVC-Aircraft

Bật GPU (T4/P100) và Internet. RTG dùng CLIP ViT-H-14 (laion2B), tải từ HuggingFace.
Kết quả: `rtg/{ds}_{ver}.pkl` + `rtg/{ds}_{ver}.json` (cấu hình, hash lexicon, thời gian chạy).

In [ ]:
REPO_URL = "https://github.com/tanq1965/gcd-text-leak.git"
BRANCH = "feat/n2-rtg"  # đổi thành main sau khi merge
REPO = "/kaggle/working/gcd-text-leak"
!git clone -b {BRANCH} {REPO_URL} {REPO}
%cd {REPO}
!pip install -q open_clip_torch ftfy regex
!python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"

In [ ]:
import os
CUB_ROOT = "/kaggle/input/datasets/wenewone/cub2002011"
CARS_ROOT = "/kaggle/input/datasets/rickyyyyyyy/torchvision-stanford-cars/stanford_cars"
AIRCRAFT_ROOT = "/kaggle/input/datasets/seryouxblaster764/fgvc-aircraft/fgvc-aircraft-2013b/fgvc-aircraft-2013b"
ROOTS = {"cub": CUB_ROOT, "scars": CARS_ROOT, "aircraft": AIRCRAFT_ROOT}
import pandas as pd
for ds, root in ROOTS.items():  # mọi ảnh trong split phải tồn tại
    df = pd.read_csv(f"splits/{ds}.csv", dtype=str)
    missing = [p for p in df.path if not os.path.exists(os.path.join(root, p))]
    print(ds, len(df), "ảnh, thiếu", len(missing), missing[:3])

## Smoke test: 256 ảnh CUB bản B (kiểm tra pipeline + đo tốc độ)

In [ ]:
!python -m src.rtg --dataset cub --version B --data-root {CUB_ROOT} --fp16 --limit 256

## N2: bản B cho 3 tập (CUB trước). Lexicon B được mã hoá 1 lần, các tập sau dùng cache.

In [ ]:
for ds in ["cub", "aircraft", "scars"]:
    !python -m src.rtg --dataset {ds} --version B --data-root {ROOTS[ds]} --fp16

## N6: bản C, D (cần lexicon của N3 đã merge; D không có tag)

In [ ]:
for ver in ["C", "D"]:
    for ds in ["cub", "aircraft", "scars"]:
        if os.path.exists(f"lexicon/{ds}/{ver}/attributes.csv"):
            !python -m src.rtg --dataset {ds} --version {ver} --data-root {ROOTS[ds]} --fp16
        else:
            print("chưa có lexicon", ds, ver)

In [ ]:
import pickle, json, glob
for p in sorted(glob.glob("rtg/*.pkl")):
    r = pickle.load(open(p, "rb")); m = json.load(open(p[:-4] + ".json"))
    print(p, len(r), m["seconds"], "no-tag:", m["images_without_tags"]); print("   ", r[0])
import shutil; shutil.make_archive("/kaggle/working/rtg_out", "zip", "rtg", ".", )
print("Tải /kaggle/working/rtg_out.zip lên Drive, ghi link vào README")